In [4]:
#  Using keras tunner  we are doing hyper parameter tunning 

import pandas as pd
from  tensorflow  import keras
from tensorflow.keras import layers
from kerastuner.tuners import RandomSearch

In [7]:
df=pd.read_csv("Real_Combine.csv")

In [8]:
df.head()


,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [9]:
X=df.iloc[:,:-1]
Y=df.iloc[:,-1]

## Hyper parameter tunning
1. how many numbers of hiden layers we should have?
2. how many numbers of neurons we should have in hidden layers?
3. learning rate

In [54]:
def build_model(hp):
    model = keras.Sequential()
    
    # Input + Hidden Layers
    for i in range(hp.Int('num_layer', 2, 6)):
        units = hp.Int(f'units_{i}', min_value=32, max_value=256, step=32)
        if i == 0:
            model.add(layers.Dense(
                units=units,
                activation='relu',
                kernel_initializer='he_normal',
                input_shape=(X_train.shape[1],)
            ))
        else:
            model.add(layers.Dense(
                units=units,
                activation='relu',
                kernel_initializer='he_normal'
            ))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(0.2))
    
    # Output Layer
    model.add(layers.Dense(1, activation='linear'))
    
    # Compile
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
        ),
        loss='mean_absolute_error',
        metrics=['mean_absolute_error']
    )
    
    return model


In [63]:
tuner=RandomSearch(
    build_model,
    objective='val_mean_absolute_error',
    max_trials=5,
    overwrite=True,
    executions_per_trial=3,
    directory='project',
    
    project_name='AIR Quality Index'
)

In [64]:
tuner.search_space_summary()

Search space summary
Default search space size: 4
num_layer (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 6, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 256, 'step': 32, 'sampling': 'linear'}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 256, 'step': 32, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


In [65]:
from sklearn.model_selection import train_test_split
X_train, X_test,Y_train, Y_test= train_test_split(X,Y, test_size=0.3, random_state=0)

In [66]:
import numpy as np
import pandas as pd

print("NaNs in X_train:", np.isnan(X_train).sum())
print("NaNs in Y_train:", np.isnan(Y_train).sum())
print("Infs in X_train:", np.isinf(X_train).sum())
print("Infs in Y_train:", np.isinf(Y_train).sum())

print("X_train mean:", np.mean(X_train))
print("X_train std:", np.std(X_train))
print("Y_train sample:", Y_train[:10])


NaNs in X_train: T      0
TM     0
Tm     0
SLP    0
H      0
VV     0
V      0
VM     0
dtype: int64
NaNs in Y_train: 1
Infs in X_train: T      0
TM     0
Tm     0
SLP    0
H      0
VV     0
V      0
VM     0
dtype: int64
Infs in Y_train: 0
X_train mean: 146.65140522875816
X_train std: T       7.241872
TM      6.689913
Tm      7.427015
SLP     7.463975
H      15.831496
VV      0.720376
V       3.677878
VM      7.112547
dtype: float64
Y_train sample: 1068    235.333333
181     166.916667
425      92.916667
1019     57.583333
9       107.625000
855      30.625000
435      77.708333
403      34.916667
751      93.375000
900     365.291667
Name: PM 2.5, dtype: float64


D:\Data-Science-Projects\venv\Lib\site-packages\numpy\_core\fromnumeric.py:4062: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [67]:
Y_train = Y_train.fillna(Y_train.mean())   # or median


In [68]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



tuner.search(X_train, Y_train, epochs=5, validation_data=(X_test,Y_test))

Trial 5 Complete [00h 00m 22s]
val_mean_absolute_error: 105.84791056315105

Best val_mean_absolute_error So Far: 102.26337432861328
Total elapsed time: 00h 01m 32s


In [70]:
tuner.results_summary()

Results summary
Results in project\AIR Quality Index
Showing 10 best trials
Objective(name="val_mean_absolute_error", direction="min")

Trial 1 summary
Hyperparameters:
num_layer: 3
units_0: 224
units_1: 32
learning_rate: 0.001
units_2: 192
units_3: 160
units_4: 32
units_5: 256
Score: 102.26337432861328

Trial 0 summary
Hyperparameters:
num_layer: 6
units_0: 64
units_1: 224
learning_rate: 0.001
units_2: 32
units_3: 32
units_4: 32
units_5: 32
Score: 105.02719370524089

Trial 4 summary
Hyperparameters:
num_layer: 6
units_0: 224
units_1: 160
learning_rate: 0.0001
units_2: 256
units_3: 96
units_4: 32
units_5: 128
Score: 105.84791056315105

Trial 3 summary
Hyperparameters:
num_layer: 3
units_0: 128
units_1: 128
learning_rate: 0.0001
units_2: 32
units_3: 96
units_4: 256
units_5: 224
Score: 105.90060933430989

Trial 2 summary
Hyperparameters:
num_layer: 5
units_0: 32
units_1: 32
learning_rate: 0.0001
units_2: 32
units_3: 128
units_4: 64
units_5: 128
Score: 105.94732920328777
